# GA4 Purchase Propensity Modeling

This notebook documents the real chronological training run on `session_level_ecommerce.csv`. The deployable model uses only fields available at session start.

## Data and leakage contract

- Unit: one unique session.
- Target: `converted`.
- Train: 2020-11-16 through 2021-01-10.
- Test: 2021-01-11 through 2021-01-31.
- Eligible: weekday, weekend, source, medium, device, OS, country.
- Excluded: browsing totals, cart/checkout/payment, purchase, revenue, engagement, and full-session duration.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path('..').resolve()
ARTIFACTS = PROJECT / 'artifacts'
metrics = json.loads((ARTIFACTS / 'model_metrics.json').read_text())
metrics['data'], metrics['split']

## Reproduce training

The command below fits a deterministic hashed logistic regression and a smoothed categorical Naive Bayes challenger using NumPy and pandas.

In [ ]:
import subprocess, sys

source_csv = PROJECT / 'data' / 'session_level_ecommerce.csv'
subprocess.run([
    sys.executable, str(PROJECT / 'src' / 'train_model.py'),
    '--input', str(source_csv), '--output', str(ARTIFACTS)
], check=True)

## Chronological model comparison

In [ ]:
comparison = pd.DataFrame(metrics['models']).T
comparison[['pr_auc', 'roc_auc', 'log_loss', 'brier_score', 'recall_at_top_10pct', 'lift_top_10pct']]

The hashed logistic model is selected by PR-AUC. Its top score decile captures 20.1% of future purchases at 2.01× lift. The modest PR-AUC and ROC-AUC show that session-start context is useful for prioritization but insufficient for autonomous targeting.

In [ ]:
lift = pd.read_csv(ARTIFACTS / 'hashed_logistic_regression_lift_by_decile.csv')
calibration = pd.read_csv(ARTIFACTS / 'hashed_logistic_regression_calibration.csv')
display(lift)
display(calibration)

## Decision

Use the score to define strata for a randomized product experiment. Do not interpret propensity lift as incremental treatment effect. The next iteration should reconstruct a fixed event-level early window so legitimate early behavior can be added without leakage.